# Set up

In [1]:
from google.colab import userdata
import os
from huggingface_hub import login
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
login(userdata.get("HF_API_KEY"))

In [2]:
!gdown 1zVrsGf7EyZIC3V2VjymzflvrazW8xfte #LST20 corpus
!kaggle competitions download -c super-ai-engineer-ss-6-word-segmentation

Downloading...
From: https://drive.google.com/uc?id=1zVrsGf7EyZIC3V2VjymzflvrazW8xfte
To: /content/Corpus-LST20.tar.gz
100% 13.6M/13.6M [00:00<00:00, 88.7MB/s]
100% 1.95M/1.95M [00:01<00:00, 1.56MB/s]



In [3]:
!tar xzf '/content/Corpus-LST20.tar.gz'
!unzip '/content/super-ai-engineer-ss-6-word-segmentation.zip'

!rm '/content/super-ai-engineer-ss-6-word-segmentation.zip'
!rm '/content/Corpus-LST20.tar.gz'

Archive:  /content/super-ai-engineer-ss-6-word-segmentation.zip
  inflating: LST20 Annotation Guideline.pdf  
  inflating: LST20 Brief Specification.pdf  
  inflating: ws_list.txt             
  inflating: ws_sample_submission.csv  
  inflating: ws_test.txt             


In [4]:
!pip install transformers datasets seqeval evaluate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


In [10]:
LST20_TRAIN_DIR = "/content/LST20_Corpus/train"   # folder with train txt files
LST20_VAL_DIR   = "/content/LST20_Corpus/eval"     # folder with val txt files
LST20_TEST_DIR = "/content/LST20_Corpus/test"
TEST_TXT       = "/content/ws_test.txt"              # competition test file (id, value)
SAMPLE_SUB_CSV  = "/content/ws_sample_submission.csv"

MODEL_NAME  = "lst-nectec/HoogBERTa"
OUTPUT_DIR  = "./hoogberta-wordseg"
MAX_LEN     = 510       # leave 2 slots for [CLS] [SEP]
STRIDE      = 128
BATCH_SIZE  = 16
EPOCHS      = 5
LR          = 2e-5

LABEL2ID = {"B_WORD": 0, "I_WORD": 1, "E_WORD": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Start

In [6]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate

In [7]:
def parse_lst20_file(filepath):
    # Try encodings in order until one works
    encodings = ["utf-8", "cp874", "tis-620", "utf-8-sig", "latin-1"]

    lines = None
    for enc in encodings:
        try:
            with open(filepath, encoding=enc) as f:
                lines = f.readlines()
            break
        except (UnicodeDecodeError, LookupError):
            continue

    if lines is None:
        print(f"WARNING: Could not decode {filepath}, skipping")
        return [], []

    sentences_chars  = []
    sentences_labels = []
    cur_chars  = []
    cur_labels = []

    for line in lines:
        line = line.rstrip("\n")

        if line.strip() == "":
            if cur_chars:
                sentences_chars.append(cur_chars)
                sentences_labels.append(cur_labels)
                cur_chars  = []
                cur_labels = []
            continue

        parts = line.split("\t")
        word  = parts[0]

        if word == "_":
            continue

        chars = list(word)
        n     = len(chars)

        if n == 1:
            cur_chars.append(chars[0])
            cur_labels.append("B_WORD")
        else:
            for i, ch in enumerate(chars):
                cur_chars.append(ch)
                if i == 0:
                    cur_labels.append("B_WORD")
                elif i == n - 1:
                    cur_labels.append("E_WORD")
                else:
                    cur_labels.append("I_WORD")

    if cur_chars:
        sentences_chars.append(cur_chars)
        sentences_labels.append(cur_labels)

    return sentences_chars, sentences_labels


def parse_lst20_dir(directory):
    all_chars, all_labels = [], []
    for fpath in sorted(Path(directory).glob("*.txt")):
        c, l = parse_lst20_file(fpath)
        all_chars.extend(c)
        all_labels.extend(l)
    print(f"  {directory}: {len(all_chars)} sentences loaded")
    return all_chars, all_labels


print("Parsing LST20...")
train_chars, train_labels = parse_lst20_dir(LST20_TRAIN_DIR)
val_chars,   val_labels   = parse_lst20_dir(LST20_VAL_DIR)
test_chars, test_labels = parse_lst20_dir(LST20_TEST_DIR)
print("Done.")

Parsing LST20...
  /content/LST20_Corpus/train: 67104 sentences loaded
  /content/LST20_Corpus/eval: 6094 sentences loaded
  /content/LST20_Corpus/test: 5733 sentences loaded
Done.


In [8]:
# ============================================================
# STEP 2 — TOKENIZE + ALIGN LABELS (sliding window for long seqs)
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(chars_list, labels_list, max_len=MAX_LEN, stride=STRIDE):
    """
    For each sentence, slide a window of max_len characters.
    Align character labels to HoogBERTa subword tokens.
    Returns a list of dicts ready for Dataset.
    """
    all_input_ids      = []
    all_attention_mask = []
    all_labels         = []

    for chars, labels in zip(chars_list, labels_list):
        # sliding window over characters
        start = 0
        while start < len(chars):
            end        = min(start + max_len, len(chars))
            chunk_chars  = chars[start:end]
            chunk_labels = labels[start:end]

            # tokenize chunk — is_split_into_words treats each char as a "word"
            enc = tokenizer(
                chunk_chars,
                is_split_into_words=True,
                truncation=True,
                max_length=max_len + 2,   # +2 for special tokens
                padding=False,
            )

            word_ids   = enc.word_ids()   # maps token → char index in chunk
            aligned    = []
            prev_wid   = None

            for wid in word_ids:
                if wid is None:           # [CLS] or [SEP]
                    aligned.append(-100)
                elif wid != prev_wid:     # first subtoken of this char
                    aligned.append(LABEL2ID[chunk_labels[wid]])
                else:                     # subsequent subtokens → ignore
                    aligned.append(-100)
                prev_wid = wid

            all_input_ids.append(enc["input_ids"])
            all_attention_mask.append(enc["attention_mask"])
            all_labels.append(aligned)

            if end == len(chars):
                break
            start += max_len - stride     # overlap

    return all_input_ids, all_attention_mask, all_labels


# STEP 3 — TOKENIZE (combine train + val first)
print("Combining train + val...")
combined_chars  = train_chars  + val_chars
combined_labels = train_labels + val_labels
print(f"  Total sentences: {len(combined_chars)}")

print("Tokenizing combined dataset...")
tr_ids, tr_mask, tr_lab = tokenize_and_align(combined_chars, combined_labels)
print(f"  {len(tr_ids)} windows")

print("Tokenizing validate dataset...")
va_ids, va_mask, va_lab = tokenize_and_align(test_chars, test_labels)
print(f"  {len(va_ids)} windows")
# no separate val tokenization needed

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

Combining train + val...
  Total sentences: 73198
Tokenizing combined dataset...
  73867 windows
Tokenizing validate dataset...
  5735 windows


In [11]:
class WordSegDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids      = input_ids
        self.attention_mask = attention_mask
        self.labels         = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      torch.tensor(self.input_ids[idx],      dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx],  dtype=torch.long),
            "labels":         torch.tensor(self.labels[idx],          dtype=torch.long),
        }


train_dataset = WordSegDataset(tr_ids, tr_mask, tr_lab)
val_dataset   = WordSegDataset(va_ids, va_mask, va_lab)
print(f"Train size: {len(train_dataset)}  Val size: {len(val_dataset)}")

Train size: 73867  Val size: 5735


In [12]:
# ============================================================
# STEP 4 — MODEL + METRICS
# ============================================================
from sklearn.metrics import f1_score
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

#seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)

    flat_preds  = []
    flat_labels = []

    for pred_row, label_row in zip(preds, labels):
        for p_tok, l_tok in zip(pred_row, label_row):
            if l_tok == -100:
                continue
            flat_preds.append(ID2LABEL[p_tok])
            flat_labels.append(ID2LABEL[l_tok])

    macro_f1 = f1_score(
        flat_labels,
        flat_preds,
        labels=["B_WORD", "I_WORD", "E_WORD"],
        average="macro",
        zero_division=0,
    )
    return {"f1_macro": macro_f1}

model.safetensors:   0%|          | 0.00/575M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: lst-nectec/HoogBERTa
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
# ============================================================
# STEP 5 — TRAINING
# ============================================================
data_collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=8)
total_steps   = (len(train_dataset) // BATCH_SIZE) * EPOCHS
warmup_steps  = int(total_steps * 0.1)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = 32,
    learning_rate=LR,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(f"{OUTPUT_DIR}/best")
print("Training complete.")

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.058315,0.048953,0.980352
2,0.042340,0.041571,0.984618
3,0.033018,0.035842,0.986250
4,0.027178,0.034101,0.987152
5,0.025273,0.034068,0.987350


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete.


In [16]:
# ============================================================
# STEP 6 — PARSE TEST TXT
# ============================================================
# The test is one long string of 37,248 characters including whitespace.
# We need to:
#   1. Read the full text preserving every character
#   2. Track which positions are whitespace (excluded from submission)
#   3. Feed only non-whitespace chars to the model

with open(TEST_TXT, encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total characters in test: {len(raw_text)}")

# Build two parallel lists:
#   all_chars     — every character (1-indexed, matches competition IDs)
#   non_ws_chars  — non-whitespace chars only
#   non_ws_ids    — their 1-based positions (these are the submission IDs)

all_chars    = list(raw_text)
non_ws_chars = []
non_ws_ids   = []

for i, ch in enumerate(all_chars):
    pos = i + 1   # 1-indexed
    if ch.strip() == "":   # whitespace of any kind
        continue
    non_ws_chars.append(ch)
    non_ws_ids.append(pos)

print(f"Non-whitespace chars: {len(non_ws_chars)}")

# Cross-check against sample submission
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
print(f"Sample submission rows: {len(sample_sub)}")
# These two numbers should match exactly

Total characters in test: 37248
Non-whitespace chars: 35182
Sample submission rows: 35182


In [17]:
# ============================================================
# STEP 7 — INFERENCE (same sliding window, now on non_ws_chars)
# ============================================================
def predict_chars(chars, model, tokenizer, max_len=MAX_LEN, stride=STRIDE, device="cuda"):
    model.eval()
    n = len(chars)

    logit_accum = np.zeros((n, 3), dtype=np.float32)
    count_accum = np.zeros(n,      dtype=np.float32)

    start = 0
    while start < n:
        end   = min(start + max_len, n)
        chunk = chars[start:end]

        enc_plain = tokenizer(
            chunk,
            is_split_into_words=True,
            truncation=True,
            max_length=max_len + 2,
        )
        word_ids_list = enc_plain.word_ids()

        enc_tensor = {
            k: torch.tensor([v]).to(device)
            for k, v in enc_plain.items()
            if k in ("input_ids", "attention_mask")
        }

        with torch.no_grad():
            out = model(**enc_tensor)

        logits = out.logits[0].cpu().numpy()   # (seq_len, 3)

        prev_wid = None
        for tok_idx, wid in enumerate(word_ids_list):
            if wid is None:
                continue
            char_pos = start + wid
            if char_pos >= n:
                continue
            if wid != prev_wid:
                logit_accum[char_pos] += logits[tok_idx]
                count_accum[char_pos] += 1
            prev_wid = wid

        if end == n:
            break
        start += max_len - stride

    safe_count = np.where(count_accum > 0, count_accum, 1)
    avg_logits = logit_accum / safe_count[:, None]
    pred_ids   = np.argmax(avg_logits, axis=-1)
    pred_ids[count_accum == 0] = LABEL2ID["B_WORD"]

    return [ID2LABEL[i] for i in pred_ids]


def fix_bio(labels):
    fixed = []
    prev  = None
    for lab in labels:
        if lab == "I_WORD" and prev not in ("B_WORD", "I_WORD"):
            lab = "B_WORD"
        elif lab == "E_WORD" and prev not in ("B_WORD", "I_WORD"):
            lab = "B_WORD"
        fixed.append(lab)
        prev = lab
    return fixed


device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Running inference...")
raw_preds = predict_chars(non_ws_chars, model, tokenizer, device=device)
preds     = fix_bio(raw_preds)
print(f"Predictions: {len(preds)}")

Running inference...
Predictions: 35182


In [18]:
# ============================================================
# STEP 8 — BUILD SUBMISSION
# ============================================================
assert len(preds) == len(non_ws_ids), \
    f"Mismatch: {len(preds)} preds vs {len(non_ws_ids)} ids"

submission = pd.DataFrame({
    "Id":        non_ws_ids,
    "Predicted": preds,
})

print(f"Submission rows: {len(submission)}")
print(submission.head(10))
print(submission["Predicted"].value_counts())

submission.to_csv("submission_wordseg_hoogberta_batch16.csv", index=False)
print("Saved.")

Submission rows: 35182
   Id Predicted
0   1    B_WORD
1   2    I_WORD
2   3    E_WORD
3   4    B_WORD
4   5    I_WORD
5   6    E_WORD
6   7    B_WORD
7   8    I_WORD
8   9    I_WORD
9  10    I_WORD
Predicted
I_WORD    19306
B_WORD     8074
E_WORD     7802
Name: count, dtype: int64
Saved.


In [19]:
from google.colab import files
files.download("submission_wordseg_hoogberta_batch16.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>